# 24 · GenAI evaluation and retrieval regression tests

**Prerequisites:** 00 — Setup; basic Python. No other notebook state is required.

**Execution:** portable. Run cells from top to bottom. Each notebook is independent.

## What you will learn

Measure retrieval Recall@k, MRR, abstention, citation validity and response schemas.


## Step 1 — Build a labeled evaluation set

Labels are independent of the retriever. This tiny set checks mechanics; use larger representative and adversarial sets for decisions.


In [ ]:
documents = [
    {'id':'delta', 'text':'Delta Lake supports ACID transactions and table version history.'},
    {'id':'catalog', 'text':'Unity Catalog controls permissions and governs tables and models.'},
    {'id':'mlflow', 'text':'MLflow tracks experiments, parameters, metrics, and model artifacts.'},
    {'id':'spark', 'text':'Spark processes large datasets using distributed DataFrame operations.'},
    {'id':'rag', 'text':'Retrieval augmented generation retrieves evidence before generating an answer.'},
    {'id':'batch', 'text':'Batch inference scores many records together on a schedule.'},
]

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
matrix = vectorizer.fit_transform([d['text'] for d in documents])
def retrieve(question, k=2):
    scores = cosine_similarity(vectorizer.transform([question]), matrix)[0]
    return [dict(documents[i], score=float(scores[i]))
            for i in np.argsort(-scores)[:k] if scores[i] > 0]

cases=[('ACID transactions','delta'),('governs permissions','catalog'),('tracks experiments','mlflow'),('distributed DataFrame','spark')]
print('Labeled queries:',len(cases))


**Check your result:** Four queries have a known relevant source.


## Step 2 — Measure ranking

Recall@k checks whether a relevant source is present. Reciprocal rank rewards placing the first relevant document early.


In [ ]:
recalls=[]; reciprocal=[]
for query,gold in cases:
    ids=[h['id'] for h in retrieve(query,2)]
    recalls.append(int(gold in ids))
    reciprocal.append(1/(ids.index(gold)+1) if gold in ids else 0)
    print(query,ids)
metrics={'recall_at_2':float(np.mean(recalls)),'mrr_at_2':float(np.mean(reciprocal))}
print(metrics)
assert metrics['recall_at_2']==1.0


**Check your result:** All four simple labels appear in the top two.


## Step 3 — Check abstention and citations

This validates an explicit citation fixture. It does not claim a generated answer is factually grounded.


In [ ]:
known_ids={d['id'] for d in documents}
fixture={'answer':'MLflow tracks experiments.','citations':['mlflow']}
assert set(fixture['citations'])<=known_ids
assert retrieve('zzzzqqqq')==[]
print('Fixture citation and lexical abstention checks passed')


**Check your result:** Known citations pass and a zero-overlap query abstains.


## Step 4 — Define quality gates

Separate retrieval quality, answer quality, latency and cost. Human-labeled support is needed to judge groundedness; word overlap cannot prove it.


In [ ]:
gates={'retrieval_recall_at_2':metrics['recall_at_2']>=0.9,
       'no_unknown_citation':set(fixture['citations'])<=known_ids}
assert all(gates.values())
print(gates)


**Check your result:** The small deterministic retrieval regression gate passes.


## Practice

Add paraphrases, unanswerable questions, malicious evidence and conflicting sources. Compare two retrieval configurations.

Predict the outcome before changing the code. Explain the result in your own words.


## Solution guide

Keep a development set for tuning and a locked evaluation set for final comparison. For LLM judges, calibrate a rubric against human labels and report disagreement.

## Before using this at work

Use representative data, measure quality on held-out examples, record dependencies and permissions, and review failures. Tiny synthetic results demonstrate mechanics; they do not establish production quality.
